# Data Preprocessing

In [1]:
# Load data from Excel file in data folder
import pandas as pd

cpds_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='COMPOUNDS')
targets_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='TARGETS')
external_ids_raw_data = pd.read_excel('../data/pd_export_02_2025_targets_standardized.xlsx', sheet_name='EXTERNAL IDS')


In [2]:
# Merge the two dataframes based on pdid column
raw_data = pd.merge(targets_raw_data, cpds_raw_data, on=['pdid', 'name'], how='left')

In [3]:
print("Cpds data shape:", cpds_raw_data.shape)
print("Targets data shape:", targets_raw_data.shape)
print("Raw data shape:", raw_data.shape)

Cpds data shape: (138689, 38)
Targets data shape: (270054, 35)
Raw data shape: (270054, 71)


### Bioactivity table

In [4]:
import re

# Lookup table that flags where each annotation came from.
assay_type = pd.DataFrame(
    [
        {
            "assay_type": "biochemical",
            "source_column": "activity_biochemical",
            "description": "Annotation extracted from activity_biochemical"
        },
        {
            "assay_type": "cell",
            "source_column": "activity_cell",
            "description": "Annotation extracted from activity_cell"
        },
    ]
)

PLACEHOLDERS = {"-", "na", "n/a", "none", "nan", "not available", "not determined"}

def split_activity_annotations(value):
    """Split one activity field into separate annotations/rows."""
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    # Most exports separate annotations by ';', '|', or newlines.
    parts = re.split(r"\s*(?:;|\||\n)+\s*", text)
    clean_parts = []
    for part in parts:
        p = part.strip()
        if p and p.lower() not in PLACEHOLDERS:
            clean_parts.append(p)

    return clean_parts

def parse_relation_value_unit(annotation):
    """Extract relation, numeric value, and unit from one annotation string."""
    relation = None
    value = None
    unit = None

    rel_match = re.search(r"(<=|>=|=|<|>)", annotation)
    if rel_match:
        relation = rel_match.group(1)

    val_match = re.search(r"(?:<=|>=|=|<|>)?\s*(-?\d+(?:\.\d+)?)\s*([a-zA-ZµμnmpkM/]+)?", annotation)
    if val_match:
        value = pd.to_numeric(val_match.group(1), errors="coerce")
        unit = val_match.group(2)

    return relation, value, unit

def parse_bioactivity_type(annotation):
    """Try to extract endpoint type (e.g., IC50, EC50, Ki) from annotation text."""
    type_match = re.match(r"\s*([A-Za-z][A-Za-z0-9_-]{1,20})\s*[:=<>]?", annotation)
    if not type_match:
        return None

    candidate = type_match.group(1).upper()
    # Avoid labeling plain numbers as types.
    if re.fullmatch(r"\d+(?:\.\d+)?", candidate):
        return None

    return candidate

records = []

for _, row in raw_data.iterrows():
    # Keep complexes unsplit in bioactivity_table (e.g., "EHMT1,EHMT2").
    target_key = row.get("gene_name")

    if pd.isna(target_key) or str(target_key).strip() == "" or str(target_key).strip().lower() in PLACEHOLDERS:
        continue

    for assay_col, assay_label in [("activity_biochemical", "biochemical"), ("activity_cell", "cell")]:
        annotations = split_activity_annotations(row.get(assay_col))

        for annotation in annotations:
            relation, numeric_value, unit = parse_relation_value_unit(annotation)

            # Keep assay_description empty when annotation is only a numeric value (+ optional relation/unit).
            has_letters = bool(re.search(r"[A-Za-z]", annotation))
            assay_description = annotation if has_letters else None

            records.append(
                {
                    "inchikey": row.get("inchikey"),
                    "target_key": str(target_key).strip(),
                    "moa": row.get("moa"),
                    "bioactivity_type": parse_bioactivity_type(annotation),
                    "relation": relation,
                    "value": numeric_value,
                    "unit": unit,
                    "assay_type": assay_label,
                    "assay_description": assay_description,
                    "cell_line": None,
                    "concentration": None,
                    "concentration_unit": None,
                    "source_db": "Probes & Drugs",
                    "source": None,
                    "source_xref": None,
                    "xref_id": None,
                }
            )

bioactivity_table = pd.DataFrame.from_records(
    records,
    columns=[
        "inchikey",
        "target_key",
        "moa",
        "bioactivity_type",
        "relation",
        "value",
        "unit",
        "assay_type",
        "assay_description",
        "cell_line",
        "concentration",
        "concentration_unit",
        "source_db",
        "source",
        "source_xref",
        "xref_id",
    ],
)

print("assay_type shape:", assay_type.shape)
display(assay_type)

print("bioactivity_table shape:", bioactivity_table.shape)
display(bioactivity_table.head(20))

assay_type shape: (2, 3)


,assay_type,source_column,description
0,biochemical,activity_biochemical,Annotation extracted from activity_biochemical
1,cell,activity_cell,Annotation extracted from activity_cell


bioactivity_table shape: (270657, 16)


,inchikey,target_key,moa,bioactivity_type,relation,value,unit,assay_type,assay_description,cell_line,concentration,concentration_unit,source_db,source,source_xref,xref_id
0,RRZVGDGTWNQAPW-UHFFFAOYSA-N,CECR2,-,None,None,5.81,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
1,RRZVGDGTWNQAPW-UHFFFAOYSA-N,BAZ2B,inhibitor;antagonist,None,None,6.77,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
2,RRZVGDGTWNQAPW-UHFFFAOYSA-N,BAZ2A,inhibitor;antagonist,None,None,6.96,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
3,RRZVGDGTWNQAPW-UHFFFAOYSA-N,SCN2A,-,None,None,5.18,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
4,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD7,-,None,None,6.40,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
5,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD9,antagonist;binding agent,None,None,7.10,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
6,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD9,antagonist;binding agent,None,None,6.95,None,cell,None,None,None,None,Probes & Drugs,None,None,None
7,WRUWGLUCNBMGPS-UHFFFAOYSA-N,EP300,-,None,None,6.11,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
8,WRUWGLUCNBMGPS-UHFFFAOYSA-N,CREBBP,-,None,None,5.82,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None
9,WRUWGLUCNBMGPS-UHFFFAOYSA-N,BRD4,-,None,None,5.12,None,biochemical,None,None,None,None,Probes & Drugs,None,None,None


### Compound table

In [5]:
# Build compound table from compounds raw data and enrich with ChEMBL from external IDs.
def normalize_chembl_values(values):
    cleaned = []
    for value in values:
        if pd.isna(value):
            continue
        parts = re.split(r"\s*(?:;|\||,)+\s*", str(value).strip())
        for part in parts:
            p = part.strip()
            if p and p != "-":
                cleaned.append(p)
    return ";".join(dict.fromkeys(cleaned)) if cleaned else None

def first_non_null(series):
    for value in series:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return None

cpd_with_ids = cpds_raw_data.copy()
cpd_with_ids["chembl_id"] = None

if "ChEMBL" in external_ids_raw_data.columns:
    merge_keys = [col for col in ["pdid", "name", "probe"] if col in cpd_with_ids.columns and col in external_ids_raw_data.columns]

    if merge_keys:
        ext_chembl = external_ids_raw_data[merge_keys + ["ChEMBL"]].copy()
        ext_chembl = ext_chembl.dropna(subset=["ChEMBL"])

        ext_chembl = (
            ext_chembl
            .groupby(merge_keys, dropna=False)["ChEMBL"]
            .apply(lambda s: normalize_chembl_values(s.tolist()))
            .reset_index(name="chembl_id")
        )

        cpd_with_ids = cpd_with_ids.merge(ext_chembl, on=merge_keys, how="left", suffixes=("", "_from_ext"))
        if "chembl_id_from_ext" in cpd_with_ids.columns:
            cpd_with_ids["chembl_id"] = cpd_with_ids["chembl_id_from_ext"]
            cpd_with_ids = cpd_with_ids.drop(columns=["chembl_id_from_ext"])

compound_table = (
    cpd_with_ids.groupby("inchikey", dropna=True, as_index=False)
    .agg(
        smiles=("smiles", first_non_null),
        chembl_id=("chembl_id", lambda s: normalize_chembl_values(s.tolist())),
        name=("name", first_non_null),
    )
)

compound_table = compound_table[["inchikey", "smiles", "chembl_id", "name"]]

print("compound_table shape:", compound_table.shape)
display(compound_table.head(20))

compound_table shape: (138689, 4)


,inchikey,smiles,chembl_id,name
0,AAAAZQPHATYWOK-JXMROGBWSA-N,CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4nc5ccccc5s4)c(Cl)...,CHEMBL175513,PD136037
1,AAADKYXUTOBAGS-XCBNKYQSSA-N,CC1(C)[C@@H]2CC[C@@]1(C)C(=S)C2,None,Thiocamphor
2,AAAFZMYJJHWUPN-SOOFDHNKSA-N,O=P(O)(O)OC[C@H]1OC(OP(=O)(O)O)[C@H](O)[C@@H]1O,None,PD199255
3,AAALVBIWOWZUBR-UHFFFAOYSA-N,Cc1cccc(NS(=O)(=O)c2ccc3c(c2)CCCN3C(=O)C2CCC2)c1C,CHEMBL1354533,PD230083
4,AAALVYBICLMAMA-UHFFFAOYSA-N,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,CHEMBL268868,DAPH
5,AAANIVVODIZQAG-NYJWCRJYSA-N,COc1cc(N2CCN(CCCCCCCCC(=O)N[C@H](C(=O)N3C[C@H]...,None,PD172935
6,AAAPMLXDGIAFDK-UHFFFAOYSA-O,CCn1c(CNC(=O)c2ncc(C)nc2N)[n+](CC)c2ccc(C(=O)N...,CHEMBL5315519,PD205517
7,AAAQFGUYHFJNHI-GOSISDBHSA-N,CCNC(=O)C[C@H]1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-n...,CHEMBL2153435,GSK 525768A
8,AAAQFGUYHFJNHI-SFHVURJKSA-N,CCNC(=O)C[C@@H]1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-...,CHEMBL1232461,molibresib
9,AAAQFGUYHFJNHI-UHFFFAOYSA-N,CCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-n2c(C...,None,molibresib


### Target table

In [6]:
# Build target table from targets raw data with unsplit target_key values.
type_candidates = ["type", "target_type", "target class", "target_class"]
type_col = next((col for col in type_candidates if col in targets_raw_data.columns), None)

if type_col is None:
    type_series = pd.Series([None] * len(targets_raw_data), index=targets_raw_data.index)
else:
    type_series = targets_raw_data[type_col]

target_table = pd.DataFrame(
    {
        "target_key": targets_raw_data.get("gene_name"),
        "type": type_series,
        "name": targets_raw_data.get("target_name"),
    }
)

target_table["target_key"] = target_table["target_key"].astype(str).str.strip()

target_table = (
    target_table
    .replace({"target_key": {"": None, "-": None, "nan": None}})
    .dropna(subset=["target_key"])
    .drop_duplicates(subset=["target_key", "type", "name"])
    .reset_index(drop=True)
)

print("target_table shape:", target_table.shape)
display(target_table.head(20))

target_table shape: (7218, 3)


,target_key,type,name
0,CECR2,single protein,Chromatin remodeling regulator CECR2
1,BAZ2B,single protein,Bromodomain adjacent to zinc finger domain pro...
2,BAZ2A,single protein,Bromodomain adjacent to zinc finger domain pro...
3,SCN2A,single protein,Sodium channel protein type 2 subunit alpha
4,BRD7,single protein,Bromodomain-containing protein 7
5,BRD9,single protein,Bromodomain-containing protein 9
6,EP300,single protein,Histone acetyltransferase p300
7,CREBBP,single protein,CREB-binding protein
8,BRD4,single protein,Bromodomain-containing protein 4
9,BRPF1,single protein,Peregrin


### Uniprot table

In [7]:
# Build Uniprot table from targets raw data with one row per gene/uniprot pair.
def split_components(value):
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    parts = [p.strip() for p in text.split(",") if p.strip()]
    return parts

uniprot_records = []

for _, row in targets_raw_data.iterrows():
    genes = split_components(row.get("gene_name"))
    uniprots = split_components(row.get("human_uniprot_id"))

    # Align gene and uniprot components by position for protein complexes.
    max_len = max(len(genes), len(uniprots))

    if max_len == 0:
        continue

    if len(genes) < max_len:
        genes = genes + [None] * (max_len - len(genes))
    if len(uniprots) < max_len:
        uniprots = uniprots + [None] * (max_len - len(uniprots))

    for i in range(max_len):
        target_key = genes[i]
        uniprot_id = uniprots[i]
        hgnc = genes[i]

        if target_key is None:
            continue

        uniprot_records.append(
            {
                "uniprot_id": uniprot_id,
                "target_key": target_key,
                "hgnc": hgnc,
                "species": "Homo sapiens",
            }
        )

uniprot_table = pd.DataFrame(
    uniprot_records,
    columns=["uniprot_id", "target_key", "hgnc", "species"],
)

uniprot_table = (
    uniprot_table
    .dropna(subset=["uniprot_id", "target_key"])
    .drop_duplicates(subset=["uniprot_id", "target_key", "hgnc"])
    .reset_index(drop=True)
)

print("uniprot_table shape:", uniprot_table.shape)
display(uniprot_table.head(20))

uniprot_table shape: (7718, 4)


,uniprot_id,target_key,hgnc,species
0,Q9BXF3,CECR2,CECR2,Homo sapiens
1,Q9UIF8,BAZ2B,BAZ2B,Homo sapiens
2,Q9UIF9,BAZ2A,BAZ2A,Homo sapiens
3,Q99250,SCN2A,SCN2A,Homo sapiens
4,Q9NPI1,BRD7,BRD7,Homo sapiens
5,Q9H8M2,BRD9,BRD9,Homo sapiens
6,Q09472,EP300,EP300,Homo sapiens
7,Q92793,CREBBP,CREBBP,Homo sapiens
8,O60885,BRD4,BRD4,Homo sapiens
9,P55201,BRPF1,BRPF1,Homo sapiens


In [8]:
from pathlib import Path

# Export all tables to TSV files.
output_dir = Path("../files")
output_dir.mkdir(parents=True, exist_ok=True)

bioactivity_path = output_dir / "bioactivity.tsv"
compound_path = output_dir / "compound.tsv"
target_path = output_dir / "target.tsv"
uniprot_path = output_dir / "uniprot.tsv"

bioactivity_table.to_csv(bioactivity_path, sep="\t", index=False)
compound_table.to_csv(compound_path, sep="\t", index=False)
target_table.to_csv(target_path, sep="\t", index=False)
uniprot_table.to_csv(uniprot_path, sep="\t", index=False)

print("Exported files:")
print(bioactivity_path)
print(compound_path)
print(target_path)
print(uniprot_path)

Exported files:
../files/bioactivity.tsv
../files/compound.tsv
../files/target.tsv
../files/uniprot.tsv
